# Лабораторная работа 1 — рабочий ноутбук

**ФИО:** Сомов Даниил Иванович

**Файлы данных:** `clients.csv`, `transactions.csv` (S25)

Рабочий ноутбук: аудит исходных данных, подготовка данных,
исследование активности клиентов и формирование групп
для маркетинговой кампании.

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

ROOT_PATH = Path().resolve().parent
DATA_PATH = ROOT_PATH / "data"
CLIENTS_RAW_PATH = DATA_PATH / "clients.csv"
TRANSACTIONS_RAW_PATH = DATA_PATH / "transactions.csv"

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

Файлы прочитаны разными параметрами: справочник — UTF-8, запятая; транзакции — Windows-1251, точка с запятой, десятичная запятая. Даты транзакций имеют смешанную точность (часть значений содержит только дату), поэтому применяется format='mixed'.

In [ ]:
df_clients = pd.read_csv(CLIENTS_RAW_PATH, encoding="utf-8")
df_tx = pd.read_csv(TRANSACTIONS_RAW_PATH, encoding="cp1251", sep=";", decimal=",")

df_tx["txn_datetime_parsed"] = pd.to_datetime(df_tx["txn_datetime"], format="mixed", errors="coerce")

In [ ]:
print(f"Размер сlients: {df_clients.shape}")
print(f"Размер transactions: {df_tx.shape}")

display(df_clients.head())
display(df_tx.head())

## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

Единица наблюдения справочника — клиент; транзакций — отдельная операция. Знаменатель долей для справочника — все строки справочника (5 680), для транзакций — все строки журнала (623 089), если явно не указано иное.

In [ ]:
print("--- КЛИЕНТЫ ---")
overview_clients = pd.DataFrame({
    "dtype": df_clients.dtypes.astype(str),
    "n_unique": df_clients.nunique(),
    "n_missing": df_clients.isna().sum(),
    "missing_%": (df_clients.isna().mean() * 100).round(2),
})
display(overview_clients)

print("\n--- ТРАНЗАКЦИИ ---")
overview_tx = pd.DataFrame({
    "dtype": df_tx.dtypes.astype(str),
    "n_unique": df_tx.nunique(),
    "n_missing": df_tx.isna().sum(),
    "missing_%": (df_tx.isna().mean() * 100).round(2),
})
display(overview_tx)

### 2.2. Дубли

В clients есть повторные client_id: часть строк полностью идентична, у части различаются город и/или сегмент. В transactions повторяется txn_id; проверяется, совпадают ли атрибуты операции и является ли различие только точностью времени.

In [ ]:
client_dups = df_clients[df_clients["client_id"].duplicated(False)].sort_values("client_id")
client_conflicts = client_dups.groupby("client_id").nunique()
client_conflict_ids = client_conflicts.index[(client_conflicts > 1).any(axis=1)] if len(client_conflicts) else []

print("Повторных client_id:", client_dups["client_id"].nunique())
print("Client IDs с конфликтующей информацией:", len(client_conflict_ids))
display(client_conflicts.loc[client_conflict_ids].head())

In [ ]:
tx_dups = df_tx[df_tx["txn_id"].duplicated(False)].sort_values("txn_id")
tx_dup_audit = tx_dups.groupby("txn_id").agg(
    rows=("txn_id","size"),
    client_n=("client_id","nunique"),
    amount_n=("amount","nunique"),
    status_n=("status","nunique"),
    raw_datetime_n=("txn_datetime","nunique"),
    parsed_datetime_n=("txn_datetime_parsed","nunique")
)

print("Повторных txn_id:", tx_dups["txn_id"].nunique())
print("Полностью повторённых строк транзакций:", df_tx.duplicated().sum())
print("txn_id с различием в исходной записи времени:", (tx_dup_audit.raw_datetime_n > 1).sum())
display(tx_dup_audit.head())

### 2.3. Соответствие данных описанию полей из задания

Проверяются категории, каналы, статусы, валюты, MCC, диапазоны сумм и даты. Необычное значение не считается ошибкой автоматически: сначала фиксируется наблюдение, затем принимается решение.

In [ ]:
print("Диапазон дат транзакций:", df_tx["txn_datetime_parsed"].min(), "—", df_tx["txn_datetime_parsed"].max())
print("Транзакции вне целевого периода (2024-01-01 .. 2025-06-30):", 
      ((df_tx["txn_datetime_parsed"] < "2024-01-01") | (df_tx["txn_datetime_parsed"] > "2025-06-30 23:59:59")).sum())

In [ ]:
print("amount < 0:", int((df_tx["amount"] < 0).sum()), "| amount == 0:", int((df_tx["amount"] == 0).sum()))

In [ ]:
for col in ["gender", "segment", "loyalty_tier", "card_type", "has_mobile_app", "marketing_opt_in", "city"]:
    display(df_clients[col].value_counts(dropna=False).to_frame("count"))

In [ ]:
for col in ["currency", "category", "channel", "status"]:
    display(df_tx[col].value_counts(False).to_frame("count"))

In [ ]:
print("МСС-коды:", sorted(df_tx["mcc_code"].unique().tolist()))
mcc_by_category = df_tx.groupby("category")["mcc_code"].nunique()
print("Количество уникальных МСС-кодов в каждой исходной категории:")
display(mcc_by_category.rename("n_unique_mcc").to_frame())
print("Категории, содержащие все 9 МСС-кодов:", 
      int((mcc_by_category == 9).sum()), "из", len(mcc_by_category))

### 2.4. Связь между таблицами

In [ ]:
orphan_tx_count = (~df_tx["client_id"].isin(df_clients["client_id"])).sum()
orphan_clients_unique = df_tx.loc[~df_tx["client_id"].isin(df_clients["client_id"]), "client_id"].nunique()

clients_without_tx = (~df_clients["client_id"].isin(df_tx["client_id"])).sum()

print(f"Транзакции с отсутствующим в справочнике client_id: {orphan_tx_count} (уникальных ID: {orphan_clients_unique})")
print(f"Клиенты из справочника без единой транзакции: {clients_without_tx}")

**Выводы по аудиту.** Перечислите найденное: что это, сколько строк,
насколько это мешает решить задачу.

- В справочнике клиентов 5 680 строк и 5 625 уникальных `client_id`.
  Обнаружено 55 повторных строк, среди которых 15 идентификаторов
  содержат конфликтующие значения. При подготовке данных для каждого
  клиента оставляется одна запись, при конфликте — первая исходная.

- В таблице транзакций 623 089 строк. Обнаружено 1 500 повторных
  идентификаторов операций: 1 255 строк являются точными дублями,
  ещё 245 записей отличаются представлением времени операции.

- 86 уникальных операций находятся за пределами заданного периода.
  Они не участвуют в дальнейшем расчёте активности.

- 375 уникальных идентификаторов клиентов из таблицы транзакций
  отсутствуют в справочнике. Такие операции невозможно связать
  с характеристиками клиента, поэтому они исключаются из анализа.

- В поле `monthly_income` обнаружены 49 значений -999.
  Они не могут интерпретироваться как реальные доходы и требуют
  отдельной обработки.

- Поле `mcc_code` не подтверждает заявленное соответствие категории:
  в каждой из девяти исходных категорий встречаются все девять
  MCC-кодов. Поэтому для категоризации операций используется
  исходное поле `category`, а MCC не применяется.

Основные проблемы не препятствуют проведению исследования,
но требуют устранения повторов, ограничения периода и проверки
связи между таблицами. Неоднозначные значения, которые невозможно
надёжно восстановить, не исправляются предположениями.

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

In [ ]:
clients = df_clients.copy()
tx = df_tx.drop(columns="txn_datetime_parsed").copy()

for col in ["birth_date", "registration_date", "card_open_date"]:
    clients[col] = pd.to_datetime(
        clients[col],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

tx["txn_datetime"] = pd.to_datetime(
    tx["txn_datetime"],
    format="mixed",
    errors="coerce"
)

tx["amount"] = pd.to_numeric(tx["amount"], errors="coerce")
tx["mcc_code"] = pd.to_numeric(tx["mcc_code"], errors="coerce")

clients["branch_code"] = (
    clients["branch_code"]
    .astype("Int64")
    .astype("string")
)

income_marker_count = int(clients["monthly_income"].eq(-999).sum())

clients.loc[
    clients["monthly_income"].eq(-999),
    "monthly_income"
] = np.nan

print("Заменено маркеров пропуска monthly_income:", income_marker_count)
print(clients[["birth_date", "registration_date", "card_open_date"]].dtypes)
print(tx[["txn_datetime", "amount"]].dtypes)
print("Тип branch_code:", clients["branch_code"].dtype)

### 3.2. Категориальные признаки

In [ ]:
city_map = {
    "москва":"Москва", "г. москва":"Москва",
    "казань":"Казань",
    "екатеринбург":"Екатеринбург", "г.екатеринбург":"Екатеринбург",
    "новосибирск":"Новосибирск", "г. новосибирск":"Новосибирск",
    "н. новгород":"Нижний Новгород", "нижний новгород":"Нижний Новгород",
    "санкт-петербург":"Санкт-Петербург", "г. санкт-петербург":"Санкт-Петербург",
    "санкт петербург":"Санкт-Петербург", "спб":"Санкт-Петербург"
}

clients["city_before"] = clients["city"]
clients["city"] = clients["city"].astype("string").str.strip().str.lower().map(city_map).fillna(clients["city"].astype("string").str.strip())

In [ ]:
category_map = {
    "продукты":"Продукты", "одежда и обувь":"Одежда и обувь",
    "транспорт":"Транспорт", "азс":"АЗС", "аптеки":"Аптеки",
    "кафе и рестораны":"Кафе и рестораны", "кафе/рестораны":"Кафе и рестораны",
    "развлечения и досуг":"Развлечения и досуг", "онлайн-сервисы":"Онлайн-сервисы",
    "связь и интернет":"Связь и интернет"
}

tx["category_before"] = tx["category"]
tx["category"] = tx["category"].astype("string").str.strip().str.lower().map(category_map).fillna(tx["category"].astype("string").str.strip())

### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

Для каждого поля с пропусками оценивается не только их количество,
но и возможная причина возникновения.

- `monthly_income`: 491 исходный пропуск и 49 значений-маркеров
  `-999`. Вероятно, доход не был указан клиентом либо отсутствовал
  при заполнении справочника. Точный механизм установить нельзя.
  Значения -999 заменяются на NaN, остальные пропуски сохраняются.

- `app_last_login`: 2 200 пропусков. Их количество совпадает
  с количеством клиентов без мобильного приложения. Это структурные
  пропуски: если приложение не установлено, дата последнего входа
  отсутствует по смыслу. Заполнять их не требуется.

- `satisfaction_score`: 3 430 пропусков. Вероятное объяснение —
  отсутствие ответа клиента на опрос удовлетворённости. Однако
  подтвердить этот механизм по имеющимся данным нельзя.
  Значения не заполняются, поскольку оценка субъективна.

Таким образом, пропуски не заполняются искусственными значениями.
Для каждого поля сохраняется исходная неопределённость, а структурные
пропуски не рассматриваются как ошибки данных.

In [ ]:
missing_profile = pd.DataFrame({
    "n_missing": df_clients.isna().sum(),
    "share_pct": (df_clients.isna().mean() * 100).round(2)
})

display(missing_profile.query("n_missing > 0"))

income_markers = int(df_clients["monthly_income"].eq(-999).sum())
income_na = int(df_clients["monthly_income"].isna().sum())
income_missing_total = income_markers + income_na

print("Без приложения:", int((df_clients.has_mobile_app == "Нет").sum()))
print("Пропущено app_last_login:", int(df_clients.app_last_login.isna().sum()))
print(
    "Пропуски satisfaction_score:",
    int(df_clients.satisfaction_score.isna().sum()),
    f"({df_clients.satisfaction_score.isna().mean():.1%})"
)
print(
    "monthly_income: исходные NaN =", income_na,
    "| маркеры -999 =", income_markers,
    "| всего проблемных значений =", income_missing_total,
    f"({income_missing_total / len(df_clients):.1%})"
)

print(
    "Проверка структурных пропусков app_last_login:",
    bool(
        (df_clients.loc[df_clients.has_mobile_app == "Нет", "app_last_login"].isna()).all()
    )
)

### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

In [ ]:
tx_audit = tx.copy()
amount_stats = tx_audit.groupby(["currency","status"]).amount.agg(
    n="count", median="median", q1=lambda x:x.quantile(.25),
    q3=lambda x:x.quantile(.75), p99=lambda x:x.quantile(.99), max="max", min="min"
)

display(amount_stats.round(2))

In [ ]:
def iqr_flags(s):
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)


tx_audit["amount_iqr_flag"] = (
    tx_audit.groupby(["currency", "status"])["amount"]
    .transform(iqr_flags)
)

print(
    "IQR-выбросы:",
    int(tx_audit.amount_iqr_flag.sum()),
    "/",
    len(tx_audit)
)

print("Выбросы по валютам и статусам:")
display(
    tx_audit.groupby(["currency", "status"])["amount_iqr_flag"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_outliers", "count": "n_operations"})
)

print("Наибольшие положительные суммы:")
display(
    tx_audit.nlargest(10, "amount")[
        ["txn_id", "client_id", "txn_datetime", "amount", "currency", "status"]
    ]
)

### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

In [ ]:
funnel = []

def add_funnel(name, frame):
    funnel.append({
        "Step": name,
        "Rows": len(frame),
        "Unique txn_id": frame.txn_id.nunique()
    })

In [ ]:
t0 = tx.copy()
add_funnel("После разбора формата", t0)

t0["_time_precision"] = t0.txn_datetime.astype(str).str.len()
t1 = (
    t0.sort_values("_time_precision", ascending=False)
      .drop_duplicates("txn_id", keep="first")
      .drop(columns="_time_precision")
)
add_funnel("После дедупликации по txn_id", t1)

t2 = t1[t1.txn_datetime.between(pd.Timestamp("2024-01-01"), pd.Timestamp("2025-06-30 23:59:59"))]
add_funnel("Внутри целевого периода", t2)

t3 = t2[t2.client_id.isin(set(clients.client_id))]
add_funnel("Существует в справочнике клиентов", t3)

t4 = t3[t3.status == "успешно"]
add_funnel("Успешные операции", t4)

display(pd.DataFrame(funnel))

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [ ]:
log_rows = [
    [
        "client_id",
        "Повторные идентификаторы; 40 полностью одинаковых строк и 15 конфликтующих ID",
        "clients.client_id",
        55,
        "55 повторных строк; 15 ID с конфликтами",
        "дедупликация",
        "одна строка на client_id",
        "Для справочника действующих клиентов нужна одна запись на клиента; при конфликте сохранена первая запись исходного файла, неопределённость раскрыта.",
    ],
    [
        "txn_id",
        "Повторные ID операций; 1 255 точных дублей и 245 различий только в точности времени",
        "transactions.txn_id",
        1500,
        "1500 повторных строк",
        "дедупликация",
        "одна строка на txn_id",
        "Один txn_id трактуется как одна операция; сохраняется наиболее точная временная отметка.",
    ],
    [
        "txn_datetime",
        "86 операций за пределами периода 2024-01-01—2025-06-30",
        "txn_datetime в расчётной выборке",
        86,
        "2023-11-27—2026-08-04",
        "исключено из расчёта",
        "2024-01-01—2025-06-30",
        "Задание явно задаёт границы наблюдения; записи сохранены в исходном источнике, но не участвуют в анализе.",
    ],
    [
        "status",
        "Отклонённые операции и возвраты не отражают успешное использование",
        "transactions.status",
        int((t3.status != "успешно").sum()),
        "отклонено / возврат",
        "исключено из метрики",
        "только успешно",
        "Метрика измеряет состоявшееся использование, а не попытки оплаты и возвратные проводки.",
    ],
    [
        "city",
        "Различия регистра, пробелов, сокращений и префикса «г.»",
        "clients.city",
        int((clients.city_before.astype(str) != clients.city.astype(str)).sum()),
        "17 исходных написаний",
        "нормализация",
        "6 канонических городов",
        "Очевидные варианты сведены к единому написанию без изменения географического смысла.",
    ],
    [
        "category",
        "Варианты написания двух категорий",
        "transactions.category",
        int((tx.category_before.astype(str) != tx.category.astype(str)).sum()),
        "Кафе/рестораны; регистр; Интернет",
        "нормализация",
        "9 категорий",
        "Нужна единая частотная аналитика по категориям.",
    ],
    [
        "amount",
        "Асимметрия и IQR-выбросы по валютам",
        "transactions.amount",
        int(tx_audit.amount_iqr_flag.sum()),
        "Суммы за пределами 1.5 IQR",
        "оставлено",
        "исходные суммы",
        "Без подтверждения ошибки/мошенничества удалять операции нельзя; суммы не входят в основную метрику.",
    ],
    [
        "mcc_code",
        "Описание обещает соответствие MCC категории, но в каждой исходной категории встречаются все 9 MCC",
        "transactions: все строки",
        len(df_tx),
        "9 категорий × 9 MCC",
        "оставлено",
        "MCC не используется для категоризации",
        "Нельзя надёжно восстановить связь категории с MCC; для анализа используется поле category.",
    ],
    [
        "branch_code",
        "Числовой тип поля-идентификатора отделения",
        "clients.branch_code",
        len(df_clients),
        "Целочисленные значения от 100 до 939",
        "приведение типа",
        "Строковый идентификатор без дополнения нулями",
        "Код отделения используется как категориальный идентификатор, "
        "а не количественная величина. Преобразован в строковый тип "
        "без изменения исходных значений.",
    ],
    [
        "monthly_income",
        "Обнаружен специальный маркер отсутствующего дохода",
        "clients.monthly_income",
        49,
        "-999",
        "замена маркера пропуска",
        "NaN",
        "Значение -999 не является реальным доходом. "
        "Маркер заменён на пропуск, чтобы он не искажал описательные статистики.",
    ],
]

cleaning_log = pd.DataFrame(
    log_rows,
    columns=[
        "field",
        "issue",
        "scope",
        "n_rows",
        "value_before",
        "action",
        "value_after",
        
        "rationale",
    ],
)

display(cleaning_log)

## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

Для возраста используются среднее и стандартное отклонение,
а также медиана и межквартильный размах. Это позволяет оценить
как типичный возраст, так и возможное влияние асимметрии распределения.

Распределение ежемесячного дохода может быть асимметричным,
поэтому в качестве основных характеристик центра и разброса
используются медиана и межквартильный размах.

` satisfaction_score ` является порядковой шкалой от 1 до 5.
Для неё используются медиана, мода и частоты отдельных оценок.
Среднее арифметическое не применяется, поскольку расстояния
между значениями порядковой шкалы нельзя считать количественно
эквивалентными.

### 5.1. Распределения количественных признаков

In [ ]:
clients["age"] = (
    (pd.Timestamp("2025-06-30") - clients.birth_date).dt.days / 365.25
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(clients["age"].dropna(), bins=25)
axes[0].set(
    title="Возраст клиентов",
    xlabel="Возраст, лет",
    ylabel="Количество клиентов"
)

axes[1].hist(clients["monthly_income"].dropna(), bins=35)
axes[1].set(
    title="Месячный доход",
    xlabel="Доход, руб.",
    ylabel="Количество клиентов"
)

plt.tight_layout()
plt.show()

age = clients["age"].dropna()
income = clients["monthly_income"].dropna()
satisfaction = clients["satisfaction_score"].dropna()

descriptive_stats = pd.DataFrame([
    {
        "Признак": "Возраст",
        "count": len(age),
        "Медиана": age.median(),
        "Среднее": age.mean(),
        "Стандартное отклонение": age.std(),
        "Q1": age.quantile(.25),
        "Q3": age.quantile(.75),
        "Межквартильный размах": age.quantile(.75) - age.quantile(.25),
        "Мода": age.mode().iloc[0]
    },
    {
        "Признак": "Месячный доход",
        "count": len(income),
        "Медиана": income.median(),
        "Среднее": np.nan,
        "Стандартное отклонение": np.nan,
        "Q1": income.quantile(.25),
        "Q3": income.quantile(.75),
        "Межквартильный размах": income.quantile(.75) - income.quantile(.25),
        "Мода": np.nan
    },
    {
        "Признак": "Удовлетворённость",
        "count": len(satisfaction),
        "Медиана": satisfaction.median(),
        "Среднее": np.nan,
        "Стандартное отклонение": np.nan,
        "Q1": satisfaction.quantile(.25),
        "Q3": satisfaction.quantile(.75),
        "Межквартильный размах": satisfaction.quantile(.75) - satisfaction.quantile(.25),
        "Мода": satisfaction.mode().iloc[0]
    }
]).set_index("Признак")

display(descriptive_stats.round(2))

print("Частоты оценок удовлетворённости:")
display(
    satisfaction.value_counts()
    .sort_index()
    .rename_axis("Оценка")
    .to_frame("Количество")
)

### 5.2. Частотные характеристики категориальных признаков

In [ ]:
cat_cols = ["city", "segment", "loyalty_tier", "card_type", "has_mobile_app", "marketing_opt_in", "income_source"]
for c in cat_cols:
    display((clients[c].value_counts(dropna=False).rename("count").to_frame()
             .assign(share_pct=lambda x: (x["count"] / len(clients) * 100).round(1))))

### 5.3. Динамика по месяцам

In [ ]:
activity_tx = t4.copy()
activity_tx["month"] = activity_tx.txn_datetime.dt.to_period("M")
monthly_total = activity_tx.groupby("month").size().reindex(pd.period_range("2024-01", "2025-06", freq="M"), fill_value=0)
monthly_avg = monthly_total / len(clients)

ax = monthly_total.plot(figsize=(12, 4), marker="o")
ax.set(title="Успешные операции по месяцам", xlabel="Месяц", ylabel="Число операций")
ax.legend()
plt.tight_layout()
plt.show()

display(monthly_total.rename("successful_operations").to_frame())

### 5.4. Разрезы и связи между признаками

In [ ]:
clients_unique = clients.drop_duplicates("client_id", keep="first").copy()
periods = pd.period_range("2024-01", "2025-06", freq="M")
monthly_client = activity_tx.assign(month=activity_tx.txn_datetime.dt.to_period("M")).groupby(["client_id", "month"]).size().unstack(fill_value=0)
monthly_client = monthly_client.reindex(index=clients_unique.client_id, columns=periods, fill_value=0)
base_cols = pd.period_range("2024-01", "2024-12", freq="M")
eval_cols = pd.period_range("2025-01", "2025-06", freq="M")

In [ ]:
metrics = pd.DataFrame(index=clients_unique.client_id)
metrics["base_avg"] = monthly_client[base_cols].mean(axis=1)
metrics["eval_avg"] = monthly_client[eval_cols].mean(axis=1)
metrics["base_total"] = monthly_client[base_cols].sum(axis=1)
metrics["eval_total"] = monthly_client[eval_cols].sum(axis=1)
metrics["base_active_months"] = (monthly_client[base_cols] > 0).sum(axis=1)
metrics["eval_active_months"] = (monthly_client[eval_cols] > 0).sum(axis=1)
metrics["drop_pct"] = np.where(metrics.base_avg > 0, 1 - metrics.eval_avg / metrics.base_avg, np.nan)
metrics["drop_abs"] = metrics.base_avg - metrics.eval_avg

profile_cols = ["segment", "loyalty_tier", "card_type", "has_mobile_app", "marketing_opt_in", "city", "branch_code", "income_source", "monthly_income", "satisfaction_score", "app_last_login", "age"]
profile = clients_unique.set_index("client_id")[profile_cols]
metrics = profile.join(metrics)

In [ ]:
for c in ["segment", "loyalty_tier", "card_type", "has_mobile_app", "city"]:
    display(metrics.groupby(c).agg(
        сount=("base_avg", "size"),
        base_monthly=("base_avg", "mean"),
        eval_monthly=("eval_avg", "mean"),
        median_drop=("drop_pct", "median"),
        share_drop_30=("drop_pct", lambda x: (x > .30).mean())
    ).round(3))

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

In [ ]:
eligible = metrics[metrics.base_avg > 0].copy()
eligible["drop_pct"] = 1 - eligible.eval_avg / eligible.base_avg

In [ ]:
print("Клиентов в справочнике:", len(metrics))
print("Клиентов с хотя бы одной успешной транзакцией в 2024 году:", len(eligible))
print("Средняя частота: 2024 =", round(metrics.base_avg.mean(), 2),
      "| янв-июн 2025 =", round(metrics.eval_avg.mean(), 2), "операций/мес.")
print("Медианное индивидуальное снижение среди клиентов с базовой активностью:",
      f"{eligible.drop_pct.median():.1%}")
print("Клиентов со снижением >30%:", int((eligible.drop_pct > 0.30).sum()))
print("Клиентов с ростом активности:", int((eligible.drop_pct < 0).sum()))

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

In [ ]:
def dim_monthly_counts(frame, dimension, label):
    p = frame.pivot_table(index="client_id", columns=["period", dimension], values="txn_id", aggfunc="count", fill_value=0)
    values = sorted(frame[dimension].dropna().unique())
    for v in values:
        b = p[("base", v)] / 12 if ("base", v) in p.columns else pd.Series(0, index=p.index)
        e = p[("eval", v)] / 6 if ("eval", v) in p.columns else pd.Series(0, index=p.index)
        metrics.loc[b.index, f"{label}_{v}_base"] = b
        metrics.loc[e.index, f"{label}_{v}_eval"] = e

activity_tx["period"] = np.where(activity_tx.txn_datetime.dt.year.eq(2024), "base", "eval")
dim_monthly_counts(activity_tx, "category", "cat")
dim_monthly_counts(activity_tx, "channel", "ch")

In [ ]:
metrics["clothes_share"] = metrics.get("cat_Одежда и обувь_base", 0) / metrics.base_avg.replace(0, np.nan)
metrics["clothes_drop"] = 1 - metrics.get("cat_Одежда и обувь_eval", 0) / metrics.get("cat_Одежда и обувь_base", 0).replace(0, np.nan)
metrics["card_share"] = metrics.get("ch_Карта_base", 0) / metrics.base_avg.replace(0, np.nan)
metrics["card_drop"] = 1 - metrics.get("ch_Карта_eval", 0) / metrics.get("ch_Карта_base", 0).replace(0, np.nan)

In [ ]:
A = (metrics.segment.eq("Массовый") & metrics.has_mobile_app.eq("Нет") & metrics.base_avg.ge(1) & metrics.drop_pct.ge(.20))
B = (metrics.clothes_share.ge(.25) & metrics.clothes_drop.ge(.40) & metrics.drop_pct.ge(.20) & metrics.base_avg.ge(2) & ~A)
C = (metrics.card_share.ge(.60) & metrics.card_drop.ge(.40) & metrics.drop_pct.ge(.20) & metrics.base_avg.ge(2) & ~A & ~B)

priority_group = pd.Series(index=metrics.index, dtype="object")
priority_group.loc[A] = "А. Массовый сегмент без мобильного приложения"
priority_group.loc[B] = "Б. Снижение объема покупок одежды"
priority_group.loc[C] = "В. Зависимость от пластиковых карт"

In [ ]:
campaign_candidates = priority_group.dropna().rename("group").to_frame().join(metrics[["marketing_opt_in"]])
campaign = campaign_candidates[campaign_candidates.marketing_opt_in.eq("Да")].reset_index(names="client_id")[["client_id", "group"]]

display(campaign.groupby("group").size().rename("count").to_frame())
print("Всего:", len(campaign), "уникальных клиентов:", campaign.client_id.nunique())

### 7.1. Кого в кампанию не берём и почему

In [ ]:
excluded = pd.DataFrame({
    "Причина":[
        "Нет согласия на маркетинговую коммуникацию",
        "Нет базовой активности в 2024 году",
        "Не попали в правила трёх приоритетных групп",
        "Не прошли лимит кампании / не входят в справочник"
    ],
    "count":[
        int((metrics.marketing_opt_in!="Да").sum()),
        int((metrics.base_avg==0).sum()),
        int((priority_group.isna()).sum()),
        0
    ]
})

display(excluded)

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

In [ ]:
hyp = pd.DataFrame([
    ["Падение одинаково во всех сегментах", metrics.groupby("segment").drop_pct.median().round(3).to_dict()],
    ["Отсутствие приложения связано с более сильным снижением", metrics.groupby("has_mobile_app").drop_pct.median().round(3).to_dict()],
    ["Казань — единственный источник падения", metrics.groupby("city").drop_pct.median().sort_values(ascending=False).round(3).to_dict()],
    ["Все каналы падают одинаково", activity_tx.groupby(["period", "channel"]).size().unstack(fill_value=0).to_dict()]
], columns=["Гипотеза", "Результат"])

pd.set_option('display.max_colwidth', None)
display(hyp)

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [ ]:
clean = metrics.reset_index(names="client_id")
clean.to_csv("../data/clean_data.csv", index=False, encoding="utf-8-sig")
cleaning_log.to_csv("../cleaning_log.csv", index=False, encoding="utf-8-sig")
campaign[["client_id", "group"]].to_csv("../campaign_list.csv", index=False, encoding="utf-8-sig")
monthly_client.reset_index(names="client_id").to_csv("../data/monthly_client_activity.csv", index=False, encoding="utf-8-sig")

In [ ]:
assert campaign["client_id"].is_unique
assert campaign["group"].nunique() == 3
assert campaign.groupby("group").size().min() >= 150
assert len(campaign) <= 2000

print("Сохранено в корень и data/: clean_data.csv, cleaning_log.csv, campaign_list.csv, monthly_client_activity.csv")
print("Контроль лимитов: n =", len(campaign), "; min group =", campaign.groupby("group").size().min())